# 03 — Executive Dashboard: Market Intelligence

Streamlit-ready notebook with trend explorer, regional maps, and breakout alerts.

**Data:** Real Google Trends (pytrends live API) | 14 keywords | 5 years


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

df_ww = pd.read_csv('data/interest_over_time_worldwide.csv', index_col=0, parse_dates=True)
df_region = pd.read_csv('data/interest_by_region_us.csv')
rising = pd.read_csv('data/related_queries_rising.csv')
print('Data loaded:', df_ww.shape, df_region.shape, rising.shape)

Data loaded: (262, 14) (714, 6) (340, 5)


## 3.1 Trend Explorer — Multi-topic Time Series

In [2]:
# Interactive multi-select trend explorer
selected = ['AI', 'ChatGPT', 'Bitcoin', 'inflation', 'crypto']
fig = go.Figure()
for kw in selected:
    if kw in df_ww.columns:
        fig.add_trace(go.Scatter(x=df_ww.index, y=df_ww[kw], name=kw, mode='lines',
                                  line=dict(width=2)))
fig.update_layout(title='Trend Explorer: Selected Topics', xaxis_title='Date', yaxis_title='Interest (0–100)',
                  hovermode='x unified', height=500, template='plotly_white')
fig.show()

## 3.2 Regional Comparison Map

In [3]:
kw = 'AI'
sub = df_region[df_region['keyword'] == kw]
fig = px.choropleth(sub, locations='geoCode', color='interest', hover_name='geoName',
                    scope='usa', color_continuous_scale='Viridis',
                    title=f'Regional Interest: "{kw}" by US State')
fig.update_layout(height=450)
fig.show()

## 3.3 Breakout Alerts — Rising Related Queries

In [4]:
# Show top rising queries by parent keyword
print('=== BREAKOUT ALERTS: Top Rising Related Queries ===')
for kw in rising['keyword'].unique()[:7]:
    sub = rising[rising['keyword'] == kw].head(3)
    if not sub.empty:
        print(f"\n🔥 {kw}:")
        for _, row in sub.iterrows():
            print(f"   • {row['query']} (+{row['value']}%)")

=== BREAKOUT ALERTS: Top Rising Related Queries ===

🔥 AI:
   • perplexity ai (+59350%)
   • perplexity (+55350%)
   • ai humanizer (+52200%)

🔥 machine learning:
   • llm machine learning (+38300%)
   • llm (+26100%)
   • best laptop 2024 (+7350%)

🔥 ChatGPT:
   • what is chatgpt (+370050%)
   • ai (+351600%)
   • chatgpt ai (+335550%)

🔥 Bitcoin:
   • meta stock (+13600%)
   • how to buy bitcoin on etoro (+12400%)
   • solana price (+11750%)

🔥 Tesla:
   • meta stock (+24600%)
   • meta stock price (+16250%)
   • tesla pi phone (+11500%)

🔥 Netflix:
   • adolescence netflix (+6400%)
   • fool me once netflix (+6050%)
   • ed gein (+5300%)

🔥 Amazon:
   • amazon affiliate program (+300%)
   • amazon kdp (+300%)
   • amazon customer service phone number (+250%)


## 3.4 Forecast Panel — 1-Year Linear Projection

In [5]:
forecast_panel = []
for kw in df_ww.columns:
    y = df_ww[kw].values
    X = np.arange(len(y)).reshape(-1, 1)
    model = LinearRegression().fit(X, y)
    future = np.array([[len(y) + i] for i in range(1, 53)])
    preds = model.predict(future)
    current = y[-4:].mean()
    forecast_panel.append({'keyword': kw, 'current_avg': round(current, 1),
                           'forecast_avg': round(preds.mean(), 1),
                           'trend': '↗ Up' if model.coef_[0] > 0.05 else ('↘ Down' if model.coef_[0] < -0.05 else '→ Flat')})
forecast_panel_df = pd.DataFrame(forecast_panel).sort_values('forecast_avg', ascending=False)
print(forecast_panel_df.to_string(index=False))

         keyword  current_avg  forecast_avg  trend
         ChatGPT         67.0          76.7   ↗ Up
              AI         76.5          73.5   ↗ Up
          Amazon         62.2          65.4 ↘ Down
          crypto         38.0          35.9 → Flat
    stock market         25.2          29.1   ↗ Up
         Netflix         25.0          27.8 → Flat
       inflation         20.2          20.2 → Flat
         fitness         11.0           8.8 → Flat
           Tesla          6.8           7.3 → Flat
         Bitcoin          5.0           6.1 → Flat
       recession          3.5           3.2 → Flat
   mental health          3.5           3.1 → Flat
machine learning          1.5           1.6 → Flat
      telehealth          0.0           0.2 → Flat


## 3.5 Category Performance Scorecard

In [6]:
cats = {
    'Tech': ['AI', 'machine learning', 'ChatGPT', 'Bitcoin', 'Tesla', 'Netflix', 'Amazon'],
    'Health': ['telehealth', 'mental health', 'fitness'],
    'Finance': ['inflation', 'recession', 'stock market', 'crypto']
}
scorecard = []
for cat, kws in cats.items():
    valid = [k for k in kws if k in df_ww.columns]
    avg = df_ww[valid].mean().mean()
    peak = df_ww[valid].max().max()
    recent = df_ww[valid].iloc[-52:].mean().mean()
    prior = df_ww[valid].iloc[-104:-52].mean().mean() if len(df_ww) >= 104 else df_ww[valid].iloc[:52].mean().mean()
    growth = ((recent - prior) / prior * 100) if prior > 0 else 0
    scorecard.append({'Category': cat, 'Avg_Interest': round(avg, 1), 'Peak': int(peak),
                      'Recent_Avg': round(recent, 1), 'YoY_Growth_%': round(growth, 1)})
scorecard_df = pd.DataFrame(scorecard)
print(scorecard_df.to_string(index=False))

fig = px.bar(scorecard_df, x='Category', y='YoY_Growth_%', color='YoY_Growth_%',
             text='YoY_Growth_%', title='Category YoY Growth %')
fig.show()

Category  Avg_Interest  Peak  Recent_Avg  YoY_Growth_%
    Tech          24.9   100        35.9          34.9
  Health           3.3    14         4.3          40.8
 Finance          17.9   100        25.2          51.0
